# NaverCafe Trend Analysis (English Version)
Extracts trending topics from community posts (excluding brand/product discussions)
using GPT-4. Results are stored weekly in the data warehouse.

> **Note:** Original pipeline processes Korean community data.
> This version uses English-transliterated keywords for portfolio readability.

**Pipeline steps:**
1. Load existing records to find latest processed date
2. Fetch new posts since last run
3. Filter out brand-related posts
4. Call GPT-4 to extract 3 trend keywords per post
5. Upload results to data warehouse

In [ ]:
# pip install snowflake-connector-python snowflake-sqlalchemy==1.5.1 openai==0.28

In [ ]:
import os
import re
import time
import pandas as pd
import datetime
import openai
import snowflake.connector
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# ── BRAND KEYWORD DICTIONARY ──────────────────────────────────────────────
# Maps each brand to known product line names and aliases (English)
# Note: Original pipeline ran on Korean community data;
# this version uses English-transliterated equivalents for readability.

BRAND_KEYWORDS = {
    'Huggies':  '|'.join(['Huggies', 'MaxDry', 'MagicDry', 'NatureMade',
                           'MagicComfort', 'NatureMadeOrganic', 'NatureSummer']),
    'Pampers':  '|'.join(['Pampers', 'Armoni', 'BabyDry', 'TouchOfNature',
                           'AirChaCha', 'NightPants']),
    'Penelope': '|'.join(['Penelope', 'MiracleAllDay', 'ThinLight']),
    'Mamipoko': '|'.join(['Mamipoko', 'AirFit', 'GoldBreathable', 'LeafGanic']),
    'Bosomi':   '|'.join(['Bosomi', 'WonderByWonder', 'MegaDry', 'RealCottonOrganic']),
    'Kindoh':   '|'.join(['Kindoh', 'UpAndPlay', 'AllDay', 'SlimFit'])
}

TARGET_BRANDS = list(BRAND_KEYWORDS.keys())

# ── LEAKAGE DETECTION KEYWORDS ────────────────────────────────────────────
# Detects posts describing diaper leakage incidents

LEAKAGE_KEYWORDS = '|'.join([
    'leaking', 'leaked', 'leaks',
    'soaked', 'soaking', 'wet',
    'dripping', 'dripped',
    'overflow', 'overflowed',
    'backflow', 'drenched',
    'soggy', 'flooded', 'seeping'
])

# ── RISK / CONTAMINATION KEYWORDS ─────────────────────────────────────────
# Detects posts mentioning safety hazards or product contamination

RISK_KEYWORDS = [
    'foreign object', 'contamination', 'rust', 'rust water',
    'adhesive', 'mold', 'mould', 'insect', 'bug', 'fly',
    'metal particle', 'metal fragment',
    'hazardous substance', 'carcinogen', 'toxic',
    'VOC', 'benzene', 'toluene', 'styrene', 'xylene', 'TVOC'
]


In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set the following environment variables before running:
#   SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT
#   SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE
#   OPENAI_API_KEY

now = datetime.now() + timedelta(hours=9)  # KST

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

In [ ]:
# ── LOAD EXISTING RECORDS TO FIND LATEST DATE ────────────────────────────

orig_columns = ['SEQ', 'DATE', 'START_WEEK', 'KEYWORD', 'TITLE',
                'CONTENTS', 'POST_NUMBER', 'URL', 'VIEWS']
orig_content = pd.DataFrame(cur.execute('SELECT * FROM TREND_TABLE').fetchall(),
                            columns=orig_columns)
latest_date = orig_content['DATE'].max()
print(f'Latest processed date: {latest_date}')

In [ ]:
# ── FETCH NEW POSTS SINCE LAST RUN ───────────────────────────────────────
# Excludes deal boards and promotional content

content_query = f"""
SELECT *
FROM COMMUNITY_POSTS_VIEW
WHERE WDATE > '{latest_date}'
AND NOT (
   BOARD_PATH IN ('Deal Board')
   OR BOARD_PATH LIKE '%HotDeal%'
   OR TITLE LIKE '%postpartum care%'
   OR TITLE LIKE '%baby products%'
   OR TITLE LIKE '%review%'
);
"""
content = cur.execute(content_query)

if not content:
    print('No new data to process.')
else:
    content_columns = ['SEQ', 'POST_NUMBER', 'BOARD_PATH', 'CAFE_NAME', 'TITLE',
                       'CONTENTS', 'MEDIA', 'AUTHOR', 'URL', 'WDATE', 'RDATE',
                       'VIEWS', 'COMMENT_CNT', 'LIKES']
    content = pd.DataFrame(content, columns=content_columns)
    content['CONTENTS'] = content['CONTENTS'].fillna('')
    content['RDATE'] = pd.to_datetime(content['RDATE']).dt.strftime('%Y-%m-%d')
    content['WDATE'] = pd.to_datetime(content['WDATE']).dt.strftime('%Y-%m-%d')
    content['YEAR'] = pd.to_datetime(content['WDATE']).dt.year
    content['MONTH'] = pd.to_datetime(content['WDATE']).dt.month

In [ ]:
# ── FILTER OUT BRAND-RELATED POSTS ───────────────────────────────────────
# Posts mentioning specific brands are excluded to capture organic consumer topics

def detect_brands(content_text, title):
    return [
        brand for brand, pattern in BRAND_KEYWORDS.items()
        if re.search(pattern, content_text, re.IGNORECASE)
        or re.search(pattern, title, re.IGNORECASE)
    ]

content['BRANDS'] = content.apply(
    lambda row: detect_brands(row['CONTENTS'], row['TITLE']), axis=1
)

# Keep only non-brand posts, top 100 by views per day
content_trend = content[content['BRANDS'].apply(lambda x: len(x) == 0)].drop(columns=['BRANDS'])
content_trend = content_trend.groupby('WDATE').apply(
    lambda x: x.nlargest(100, 'VIEWS')
).reset_index(drop=True)
print(f'Posts selected for trend analysis: {len(content_trend)}')

In [ ]:
# ── GPT-4 TREND KEYWORD EXTRACTION ──────────────────────────────────────
# Extracts 3 trending topic keywords per post, excluding diaper/parenting topics

openai.api_key = os.environ['OPENAI_API_KEY']
GPT_MODEL = 'gpt-4-turbo-2024-04-09'

def extract_trend(title_, content_):
    """Call GPT-4 to extract 3 trending keywords from a community post."""
    prompt = (
        f'Title: {title_}\nContent: {content_}\n'
        'Excluding topics about diapers, pregnancy, childbirth, and parenting, '
        'identify 3 trending keywords from this post. '
        'Output format: {"new": [keyword1, keyword2, keyword3]}. No other output.'
    )
    messages = [
        {'role': 'system', 'content': 'You are a data analyst specializing in trend analysis.'},
        {'role': 'user', 'content': prompt}
    ]
    try:
        response = openai.ChatCompletion.create(model=GPT_MODEL, messages=messages)
        return response.choices[0].message['content']
    except Exception as e:
        print(f'API error: {e}')
        return '{}'

In [ ]:
# ── BATCH KEYWORD EXTRACTION ─────────────────────────────────────────────

col_seq, col_word, col_date = [], [], []
col_title, col_contents, col_post_number, col_url, col_views = [], [], [], [], []

for _, row in content_trend.iterrows():
    try:
        str_d = extract_trend(row['TITLE'], row['CONTENTS'])
        d = eval(str_d)
        for keyword in d['new']:
            col_seq.append(row['SEQ'])
            col_word.append(keyword)
            col_date.append(row['WDATE'])
            col_title.append(row['TITLE'])
            col_contents.append(row['CONTENTS'])
            col_post_number.append(row['POST_NUMBER'])
            col_url.append(row['URL'])
            col_views.append(row['VIEWS'])
    except Exception as e:
        print(f'Failed: {e}')
        time.sleep(60)

In [ ]:
# ── BUILD OUTPUT AND UPLOAD ───────────────────────────────────────────────

output = pd.DataFrame({
    'SEQ': col_seq,
    'DATE': pd.to_datetime(col_date).strftime('%Y-%m-%d'),
    'KEYWORD': col_word,
    'TITLE': col_title,
    'CONTENTS': col_contents,
    'POST_NUMBER': col_post_number,
    'URL': col_url,
    'VIEWS': col_views
})

# Exclude brand names from keyword results
output = output[~output['KEYWORD'].isin(TARGET_BRANDS)]

# Add week start date (Sunday-based)
output['START_WEEK'] = (
    pd.to_datetime(output['DATE'])
    - pd.to_timedelta(pd.to_datetime(output['DATE']).dt.weekday, unit='d')
).dt.strftime('%Y-%m-%d')

output = output[output['CONTENTS'] != '']
output['KEYWORD'] = output['KEYWORD'].str.strip()
output = output[['SEQ', 'DATE', 'START_WEEK', 'KEYWORD', 'TITLE',
                 'CONTENTS', 'POST_NUMBER', 'URL', 'VIEWS']]

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

with engine.connect() as con:
    output.to_sql(name='TREND_TABLE', con=con, if_exists='append',
                  method=pd_writer, index=False)

print(f'Uploaded {len(output)} trend keywords')